# Customer Churn Prediction - Complete Pipeline
Cleaning -> EDA -> Feature Selection -> Split/Scale/SMOTE -> Model Comparison -> Final Evaluation -> Save

Dataset: Telco Customer Churn (Kaggle, `blastchar/telco-customer-churn`). Set `DATA_PATH` below.

## 1. Imports & Load

In [ ]:
import pandas as pd, numpy as np, joblib
import seaborn as sns, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, classification_report, confusion_matrix)
from imblearn.over_sampling import SMOTE

DATA_PATH = "WA_Fn-UseC_-Telco-Customer-Churn.csv"   # <- change if needed
df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

## 2. Data Cleaning
`TotalCharges` is stored as text because 11 rows are blank (new customers with tenure 0).

In [ ]:
df.info()
print("Duplicates:", df.duplicated().sum())

In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
print("Missing TotalCharges:", df["TotalCharges"].isnull().sum())
df["TotalCharges"] = df["TotalCharges"].fillna(0)          # tenure = 0 -> nothing billed yet

df = df.drop("customerID", axis=1)                          # unique ID, no predictive value
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

# "No internet service" / "No phone service" repeat info already in InternetService / PhoneService
df = df.replace({"No internet service": "No", "No phone service": "No"})
df.isnull().sum().sum()

## 3. EDA

In [ ]:
print(df["Churn"].value_counts(normalize=True))
sns.countplot(x="Churn", data=df); plt.title("Churn balance (imbalanced)"); plt.show()

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
for a, col in zip(ax, ["tenure", "MonthlyCharges", "TotalCharges"]):
    sns.histplot(data=df, x=col, hue="Churn", kde=True, ax=a)
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
for a, col in zip(ax, ["Contract", "InternetService", "PaymentMethod"]):
    sns.countplot(data=df, x=col, hue="Churn", ax=a); a.tick_params(axis="x", rotation=20)
plt.tight_layout(); plt.show()

## 4. Encode categoricals

In [ ]:
df = pd.get_dummies(df, drop_first=True)
df = df.astype({c: int for c in df.select_dtypes("bool").columns})
print(df.shape)
df.head()

## 5. Feature Selection (beginner-friendly: 3 checks)
**Check 1 - correlation heatmap**

In [ ]:
plt.figure(figsize=(14, 9))
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap="coolwarm"); plt.show()
df.corr()["Churn"].sort_values(ascending=False)

`tenure` vs `TotalCharges` = 0.83 -> redundant (`TotalCharges ~ tenure x MonthlyCharges`).  
`tenure` is the stronger predictor of Churn, so drop **only `TotalCharges`**.

In [ ]:
df = df.drop("TotalCharges", axis=1)

**Check 2 - Random Forest importance**

In [ ]:
X = df.drop("Churn", axis=1); y = df["Churn"]
rf = RandomForestClassifier(random_state=42).fit(X, y)
importance = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
importance.plot(kind="bar", figsize=(12, 5)); plt.show()
importance

**Check 3 - drop features weak on BOTH correlation and importance**

In [ ]:
weak = ["PhoneService_Yes", "PaymentMethod_Mailed check", "PaymentMethod_Credit card (automatic)"]
df = df.drop(columns=[c for c in weak if c in df.columns])
X = df.drop("Churn", axis=1).astype(float); y = df["Churn"]
print("Final feature count:", X.shape[1])
joblib.dump(list(X.columns), "model_columns.pkl")   # needed by the API

## 6. Train/Test split -> Scale -> SMOTE
Order matters: split first, then fit scaler & SMOTE on **train only** to avoid leakage.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

num_cols = ["tenure", "MonthlyCharges"]
scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

X_train_bal, y_train_bal = SMOTE(random_state=42).fit_resample(X_train, y_train)
print("Before SMOTE:", y_train.value_counts().to_dict())
print("After  SMOTE:", y_train_bal.value_counts().to_dict())

## 7. Model comparison
Accuracy is misleading on imbalanced data - focus on **Recall / F1 / ROC-AUC**.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "KNN": KNeighborsClassifier(),
}
rows = []
for name, m in models.items():
    m.fit(X_train_bal, y_train_bal)
    pred = m.predict(X_test); proba = m.predict_proba(X_test)[:, 1]
    rows.append({"Model": name,
                 "Accuracy": accuracy_score(y_test, pred),
                 "Precision": precision_score(y_test, pred),
                 "Recall": recall_score(y_test, pred),
                 "F1": f1_score(y_test, pred),
                 "ROC-AUC": roc_auc_score(y_test, proba)})
results = pd.DataFrame(rows).sort_values("F1", ascending=False).reset_index(drop=True)
results.round(3)

## 8. Final model: Logistic Regression
Chosen for the best Recall/F1 - missing a churner costs more than a false alarm.

In [ ]:
final_model = models["Logistic Regression"]
y_pred = final_model.predict(X_test)
print(classification_report(y_test, y_pred, target_names=["No Churn", "Churn"]))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["No Churn", "Churn"], yticklabels=["No Churn", "Churn"])
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Confusion Matrix - Logistic Regression"); plt.show()

## 9. (Optional) Hyperparameter tuning

In [ ]:
from sklearn.model_selection import GridSearchCV
grid = GridSearchCV(LogisticRegression(max_iter=1000), {"C": [0.01, 0.1, 1, 10, 100]}, cv=5, scoring="f1")
grid.fit(X_train_bal, y_train_bal)
print("Best params:", grid.best_params_, "| CV F1:", round(grid.best_score_, 3))
print(classification_report(y_test, grid.best_estimator_.predict(X_test)))

## 10. Save artifacts for the API
Use the tuned model here only if it actually beat the baseline on the test set.

In [ ]:
joblib.dump(final_model, "churn_model.pkl")
joblib.dump(scaler, "scaler.pkl")
print("Saved: churn_model.pkl, scaler.pkl, model_columns.pkl")